In [5]:
import json
import os
from collections import defaultdict, Counter

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

INPUT_FILE = "./unified_logs.ndjson"   # NDJSON
OUTPUT_DIR = "analysis/regex_candidates"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# DBSCAN parameters
EPS = 0.3
MIN_SAMPLES = 30

# How many samples to show per cluster
MAX_EXAMPLES = 50

# =====================================================
# STEP 1: LOAD UNLABELED LOGS ONLY
# =====================================================

logs = []
messages = []

with open(INPUT_FILE) as f:
    for line in f:
        log = json.loads(line)

        # Only logs NOT already handled by regex
        if log.get("label") is None:
            msg = log.get("clean_message")
            if msg:
                logs.append(log)
                messages.append(msg)

print(f"[INFO] Unlabeled logs loaded: {len(messages)}")

if len(messages) < MIN_SAMPLES:
    raise RuntimeError("Not enough unlabeled logs to justify DBSCAN.")

# =====================================================
# STEP 2: EMBEDDINGS (SEMANTIC, NOT STRING)
# =====================================================

print("[INFO] Loading embedding model...")
model = SentenceTransformer(EMBEDDING_MODEL)

print("[INFO] Encoding log messages...")
embeddings = model.encode(
    messages,
    show_progress_bar=True,
    normalize_embeddings=True
)

# =====================================================
# STEP 3: DBSCAN — FIND REPEATED PATTERNS
# =====================================================

print("[INFO] Running DBSCAN...")
dbscan = DBSCAN(
    eps=EPS,
    min_samples=MIN_SAMPLES,
    metric="cosine"
)

cluster_ids = dbscan.fit_predict(embeddings)

# =====================================================
# STEP 4: GROUP CLUSTERS
# =====================================================

clusters = defaultdict(list)

for idx, cid in enumerate(cluster_ids):
    clusters[cid].append(logs[idx])

num_real_clusters = len([c for c in clusters if c != -1])

print(f"[INFO] Found {num_real_clusters} candidate clusters")

# =====================================================
# STEP 5: SAVE ONLY REGEX-WORTHY CLUSTERS
# =====================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

for cid, cluster_logs in clusters.items():
    if cid == -1:
        continue  # noise → BERT/LLM handles this

    freq = len(cluster_logs)

    # Explicit frequency gate (your requirement)
    if freq < MIN_SAMPLES:
        continue

    out_path = os.path.join(
        OUTPUT_DIR, f"cluster_{cid}_freq_{freq}.txt"
    )

    with open(out_path, "w") as f:
        for log in cluster_logs[:MAX_EXAMPLES]:
            f.write(log["clean_message"] + "\n")

    print(
        f"[CANDIDATE] cluster={cid} "
        f"freq={freq} → {out_path}"
    )

print("[DONE] DBSCAN regex pattern mining complete.")


[INFO] Unlabeled logs loaded: 10000
[INFO] Loading embedding model...
[INFO] Encoding log messages...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

[INFO] Running DBSCAN...
[INFO] Found 37 candidate clusters
[CANDIDATE] cluster=0 freq=36 → analysis/regex_candidates/cluster_0_freq_36.txt
[CANDIDATE] cluster=1 freq=39 → analysis/regex_candidates/cluster_1_freq_39.txt
[CANDIDATE] cluster=2 freq=102 → analysis/regex_candidates/cluster_2_freq_102.txt
[CANDIDATE] cluster=3 freq=97 → analysis/regex_candidates/cluster_3_freq_97.txt
[CANDIDATE] cluster=4 freq=46 → analysis/regex_candidates/cluster_4_freq_46.txt
[CANDIDATE] cluster=5 freq=91 → analysis/regex_candidates/cluster_5_freq_91.txt
[CANDIDATE] cluster=6 freq=131 → analysis/regex_candidates/cluster_6_freq_131.txt
[CANDIDATE] cluster=7 freq=35 → analysis/regex_candidates/cluster_7_freq_35.txt
[CANDIDATE] cluster=8 freq=90 → analysis/regex_candidates/cluster_8_freq_90.txt
[CANDIDATE] cluster=9 freq=69 → analysis/regex_candidates/cluster_9_freq_69.txt
[CANDIDATE] cluster=10 freq=37 → analysis/regex_candidates/cluster_10_freq_37.txt
[CANDIDATE] cluster=11 freq=57 → analysis/regex_candid

In [6]:
"hi"

'hi'

In [7]:
import pandas as pd

df = pd.read_json('./regex_output.ndjson', lines=True)


In [8]:
df.head()

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,confidence,classified_by
0,mac,Jul 1 09:00:55 calvisitor-10-105-160-95 kerne...,iothunderboltswitch<<NUM>>(0x0)::listenercallb...,INFO,kernel,2026-07-01 09:00:55,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",None,0.0,regex
1,mac,Jul 1 09:01:05 calvisitor-10-105-160-95 com.a...,thermal pressure state: <NUM> memory pressure ...,WARN,com.apple.CDScheduler,2026-07-01 09:01:05,"{'host': 'calvisitor-10-105-160-95', 'pid': '43'}",None,0.0,regex
2,mac,Jul 1 09:01:06 calvisitor-10-105-160-95 QQ[10...,fa||url||taskid[<NUM>] dealloc,INFO,QQ,2026-07-01 09:01:06,"{'host': 'calvisitor-10-105-160-95', 'pid': '1...",None,0.0,regex
3,mac,Jul 1 09:02:26 calvisitor-10-105-160-95 kerne...,arpt: <NUM>.<NUM>: airport_brcm43xx::syncpower...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",None,0.0,regex
4,mac,Jul 1 09:02:26 authorMacBook-Pro kernel[0]: A...,arpt: <NUM>.<NUM>: airport_brcm43xx::platformw...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'authorMacBook-Pro', 'pid': '0', 'com...",None,0.0,regex


In [11]:
df.classified_by.value_counts()

,count
classified_by,
regex,10000


In [20]:
df[df['confidence']>0].head()

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,confidence,classified_by
2000,linux,Jun 14 15:16:01 combo sshd(pam_unix)[19939]: a...,authentication failure; logname= uid=<NUM> eui...,ERROR,sshd,2026-06-14 15:16:01,"{'host': 'combo', 'pid': '19939', 'rhost': '21...",AUTH_FAILURE,0.95,regex
2001,linux,Jun 14 15:16:02 combo sshd(pam_unix)[19937]: c...,check pass; user unknown,WARN,sshd,2026-06-14 15:16:02,"{'host': 'combo', 'pid': '19937'}",UNKNOWN_USER_LOGIN,0.96,regex
2002,linux,Jun 14 15:16:02 combo sshd(pam_unix)[19937]: a...,authentication failure; logname= uid=<NUM> eui...,ERROR,sshd,2026-06-14 15:16:02,"{'host': 'combo', 'pid': '19937', 'rhost': '21...",AUTH_FAILURE,0.95,regex
2003,linux,Jun 15 02:04:59 combo sshd(pam_unix)[20882]: a...,authentication failure; logname= uid=<NUM> eui...,ERROR,sshd,2026-06-15 02:04:59,"{'host': 'combo', 'pid': '20882', 'user': 'roo...",AUTH_FAILURE,0.95,regex
2004,linux,Jun 15 02:04:59 combo sshd(pam_unix)[20884]: a...,authentication failure; logname= uid=<NUM> eui...,ERROR,sshd,2026-06-15 02:04:59,"{'host': 'combo', 'pid': '20884', 'user': 'roo...",AUTH_FAILURE,0.95,regex


In [22]:
df[df['confidence']>0].classified_by.value_counts()

,count
classified_by,
regex,1330
